In [42]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.optim.swa_utils import AveragedModel, SWALR

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import random

In [43]:
# set the same seed for different random generators
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [44]:
BATCH_SIZE = 128
NUM_WORKERS = 2
DATA_ROOT = "/content/data"
VAL_SIZE = 5000
PIN_MEMORY = torch.cuda.is_available()

In [45]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

raw_transform = transforms.Compose([
    transforms.ToTensor()
])

full_train_aug_dataset = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True, download=False, transform=train_transform)
full_train_raw_dataset = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True, download=False, transform=raw_transform)
test_dataset = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=raw_transform)

CIFAR10_CLASSES = full_train_raw_dataset.classes

In [46]:
num_train = len(full_train_raw_dataset)
indices = torch.randperm(num_train, generator=torch.Generator().manual_seed(RANDOM_SEED))

val_indices = indices[:VAL_SIZE]
train_indices = indices[VAL_SIZE:]

train_dataset = torch.utils.data.Subset(full_train_aug_dataset, train_indices)
val_dataset = torch.utils.data.Subset(full_train_raw_dataset, val_indices)
mean_std_dataset = torch.utils.data.Subset(full_train_raw_dataset, train_indices)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
mean_std_loader = torch.utils.data.DataLoader(mean_std_dataset, batch_size=512, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

In [47]:
class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        mean = torch.tensor(mean).view(1, 3, 1, 1)
        std = torch.tensor(std).view(1, 3, 1, 1)
        self.register_buffer("mean", mean)
        self.register_buffer("std", std)

    def forward(self, x):
        return (x - self.mean) / self.std

def compute_mean_std(loader):
    channels_sum = torch.zeros(3)
    channels_squared_sum = torch.zeros(3)
    num_pixels = 0
    for images, _ in loader:
        batch_size, channels, height, width = images.shape
        num_pixels += batch_size * height * width
        channels_sum += images.sum(dim=[0, 2, 3])
        channels_squared_sum += (images ** 2).sum(dim=[0, 2, 3])

    mean = channels_sum / num_pixels
    std = torch.sqrt(channels_squared_sum / num_pixels - mean ** 2)
    return mean.tolist(), std.tolist()

cifar10_mean, cifar10_std = compute_mean_std(mean_std_loader)

In [48]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_channels = 16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, 3, 1)
        self.layer2 = self._make_layer(32, 3, 2)
        self.layer3 = self._make_layer(64, 3, 2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)
        self._initialize_weights()

    def _make_layer(self, out_channels, num_blocks, stride):
        layers = []
        layers.append(BasicBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(self.in_channels, out_channels, 1))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.01)
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer3(self.layer2(self.layer1(out)))
        out = torch.flatten(self.avg_pool(out), 1)
        return self.fc(out)

def build_resnet20_model(cifar10_mean, cifar10_std, device, num_classes=10):
    resnet20 = ResNet20(num_classes=num_classes)
    return nn.Sequential(Normalize(cifar10_mean, cifar10_std), resnet20).to(device)

model = build_resnet20_model(cifar10_mean, cifar10_std, device, 10)

In [49]:
def cw_l2_attack(model, images, labels, device, num_classes=10, max_iters=100, learning_rate=0.01, c=1e-3, kappa=0.0):
    model.eval()
    images = images.to(device)
    labels = labels.to(device)

    # Transform images to an unbounded representation using atanh.
    # Clamping avoids inf gradients.
    images_atanh = torch.atanh(torch.clamp(images * 2 - 1, min=-0.9999, max=0.9999))
    w = images_atanh.clone().detach().requires_grad_(True)

    # We use Adam purely to optimize the perturbation tensor 'w'
    optimizer = torch.optim.Adam([w], lr=learning_rate)

    for step in range(max_iters):
        optimizer.zero_grad()

        # Convert perturbation back to valid [0, 1] image space
        adv_images = 0.5 * (torch.tanh(w) + 1.0)
        logits = model(adv_images)

        # 1. C&W Margin Loss
        one_hot_labels = F.one_hot(labels, num_classes=num_classes)
        real = torch.sum(one_hot_labels * logits, dim=1)
        # Find the max logit of the incorrect classes
        other = torch.max((1.0 - one_hot_labels) * logits - one_hot_labels * 1e4, dim=1)[0]

        # We want other > real + kappa
        loss_cw = torch.clamp(other - real + kappa, min=0.0)

        # 2. L2 Distance Penalty
        loss_l2 = F.mse_loss(adv_images, images, reduction='none').view(images.size(0), -1).sum(1)

        # Total optimization objective
        loss = torch.mean(loss_l2 + c * loss_cw)

        loss.backward()
        optimizer.step()

    return (0.5 * (torch.tanh(w) + 1.0)).detach()

In [50]:
def trades_loss(model, x, y, optimizer, step_size, epsilon, perturb_steps, beta, device, inner_loss="ce"):
    """
    Computes the TRADES loss with a toggle for the inner maximization loss.
    """
    model.eval()

    # 1. Inner Maximization (Find Adversarial Example)
    # Random start
    x_adv = x.detach() + 0.001 * torch.randn(x.shape).to(device).detach()
    x_adv = torch.clamp(x_adv, 0.0, 1.0)

    for _ in range(perturb_steps):
        x_adv.requires_grad_()
        with torch.enable_grad():
            logits = model(x_adv)

            if inner_loss == "cw":
                # CW Margin Loss - maximize margin to push misclassification
                one_hot_labels = F.one_hot(y, num_classes=10)
                real = torch.sum(one_hot_labels * logits, dim=1)
                other = torch.max((1.0 - one_hot_labels) * logits - one_hot_labels * 1e4, dim=1)[0]
                loss_inner = torch.sum(other - real)
            else:
                # Standard Cross-Entropy Loss
                loss_inner = F.cross_entropy(logits, y)

        grad = torch.autograd.grad(loss_inner, [x_adv])[0]

        # PGD step
        x_adv = x_adv.detach() + step_size * torch.sign(grad.detach())
        x_adv = torch.min(torch.max(x_adv, x - epsilon), x + epsilon)
        x_adv = torch.clamp(x_adv, 0.0, 1.0)

    # 2. Outer Minimization (Update Model)
    model.train()
    optimizer.zero_grad()

    # Normal CE loss on clean inputs
    logits_clean = model(x)
    loss_ce = F.cross_entropy(logits_clean, y)

    # Robust Regularization on adversarial inputs (KL Divergence)
    logits_adv = model(x_adv)
    loss_kl = F.kl_div(F.log_softmax(logits_adv, dim=1),
                       F.softmax(logits_clean, dim=1),
                       reduction='batchmean')

    # Total TRADES loss
    loss = loss_ce + beta * loss_kl
    return loss, logits_clean

def get_epsilon(epoch, final_eps=8/255, warmup_epochs=5):
    """Gradually ramps up epsilon to stabilize early training"""
    if epoch < warmup_epochs:
        return final_eps * ((epoch + 1) / warmup_epochs)
    return final_eps

In [51]:
def train_trades_epoch(model, loader, optimizer, epoch, beta, device, inner_loss="ce"):
    total_loss, total_correct, total_samples = 0.0, 0, 0
    current_eps = get_epsilon(epoch)

    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        # Pass the inner_loss argument here
        loss, logits = trades_loss(model, images, labels, optimizer,
                                   step_size=2/255, epsilon=current_eps,
                                   perturb_steps=10, beta=beta, device=device,
                                   inner_loss=inner_loss)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples

@torch.no_grad()
def evaluate_clean(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples

def evaluate_robust(model, loader, device, eval_batches=10):
    """
    Evaluates robustness using our C&W L2 attack on a subset of the validation set
    (to keep epoch times reasonable). Used for early stopping.
    """
    total_correct = 0
    total_samples = 0

    for idx, (images, labels) in enumerate(loader):
        if idx >= eval_batches: break # Only evaluate on a subset to save time

        adv_images = cw_l2_attack(model, images, labels, device, max_iters=50) # fewer iters for fast eval

        with torch.no_grad():
            logits = model(adv_images)
            preds = logits.argmax(dim=1)

        total_correct += (preds == labels.to(device)).sum().item()
        total_samples += labels.size(0)

    return total_correct / total_samples

In [52]:
EPOCHS = 30
LR = 0.1
BETA = 6.0 # 1/lambda = 6
SWA_START = 20
SAVE_PATH = "resnet20_cifar10_trades_cw.pth"

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)

# Stochastic Weight Averaging
swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=0.05)

history = {"train_loss": [], "train_acc": [], "val_clean_acc": [], "val_robust_acc": []}
best_robust_acc = 0.0

if not os.path.exists(SAVE_PATH):
    for epoch in range(EPOCHS):
        start_time = time.time()

        train_loss, train_acc = train_trades_epoch(model, train_loader, optimizer, epoch, BETA, device)
        val_loss, val_clean_acc = evaluate_clean(model, val_loader, criterion, device)
        val_robust_acc = evaluate_robust(model, val_loader, device, eval_batches=5)

        # SWA scheduling logic
        if epoch >= SWA_START:
            swa_model.update_parameters(model)
            swa_scheduler.step()
        else:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_clean_acc"].append(val_clean_acc)
        history["val_robust_acc"].append(val_robust_acc)

        # Robust Early Stopping
        if val_robust_acc > best_robust_acc:
            best_robust_acc = val_robust_acc
            torch.save(model.state_dict(), SAVE_PATH)

        elapsed = time.time() - start_time
        print(f"Epoch [{epoch + 1:02d}/{EPOCHS}] | "
              f"Loss: {train_loss:.4f} | "
              f"Acc: {train_acc:.4f} | "
              f"Val Clean: {val_clean_acc:.4f} | "
              f"Val Robust: {val_robust_acc:.4f} | "
              f"Time: {elapsed:.1f}s")

    print(f"Training Complete. Best Robust Acc: {best_robust_acc:.4f}")
else:
    print("Saved model found. Skipping training phase.")

Epoch [01/30] | Loss: 1.7374 | Acc: 0.3657 | Val Clean: 0.4678 | Val Robust: 0.4875 | Time: 86.3s
Epoch [02/30] | Loss: 1.5568 | Acc: 0.4892 | Val Clean: 0.5138 | Val Robust: 0.4969 | Time: 88.3s
Epoch [03/30] | Loss: 1.5524 | Acc: 0.5271 | Val Clean: 0.5182 | Val Robust: 0.5344 | Time: 88.5s
Epoch [04/30] | Loss: 1.5890 | Acc: 0.5333 | Val Clean: 0.5472 | Val Robust: 0.5500 | Time: 88.2s
Epoch [05/30] | Loss: 1.6395 | Acc: 0.5304 | Val Clean: 0.5536 | Val Robust: 0.5422 | Time: 88.3s
Epoch [06/30] | Loss: 1.6156 | Acc: 0.5461 | Val Clean: 0.5632 | Val Robust: 0.5609 | Time: 88.2s
Epoch [07/30] | Loss: 1.5987 | Acc: 0.5506 | Val Clean: 0.5506 | Val Robust: 0.5469 | Time: 88.1s
Epoch [08/30] | Loss: 1.5851 | Acc: 0.5616 | Val Clean: 0.5558 | Val Robust: 0.5453 | Time: 88.9s
Epoch [09/30] | Loss: 1.5729 | Acc: 0.5662 | Val Clean: 0.5614 | Val Robust: 0.5484 | Time: 88.3s
Epoch [10/30] | Loss: 1.5589 | Acc: 0.5759 | Val Clean: 0.5734 | Val Robust: 0.5797 | Time: 88.4s
Epoch [11/30] | Loss

In [53]:
# If training finished, SWA models need a BN update.
# Here, we ensure we load the best checkpoint saved by robust early stopping.
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()
print("Loaded optimal TRADES model.")

Loaded optimal TRADES model.


In [54]:
# Evaluate the finalized defense explicitly against the L2 C&W Attack
test_loss, test_clean_acc = evaluate_clean(model, test_loader, criterion, device)
print(f"Final Test Clean Accuracy: {test_clean_acc:.4f}")

print("Running Full L2 C&W Evaluation (First 5 batches)...")
test_robust_acc = evaluate_robust(model, test_loader, device, eval_batches=5)
print(f"Final Test C&W Robust Accuracy: {test_robust_acc:.4f}")

Final Test Clean Accuracy: 0.6715
Running Full L2 C&W Evaluation (First 5 batches)...
Final Test C&W Robust Accuracy: 0.6922
